In [1]:
import pandas as pd
import numpy as np
import sqlite3

sales_raw = pd.read_csv("data/sales_train_evaluation.csv")
calendar = pd.read_csv("data/calendar.csv")
prices = pd.read_csv("data/sell_prices.csv")

STORES = ["CA_1", "CA_2", "CA_3"]
day_cols = [c for c in sales_raw.columns if c.startswith("d_")]

# top 100 items by total CA sales
ca = sales_raw[sales_raw["store_id"].isin(STORES)].copy()
ca["total"] = ca[day_cols].sum(axis=1)
top_items = (ca.groupby("item_id")["total"].sum()
               .nlargest(100).index.tolist())
sub = ca[ca["item_id"].isin(top_items)]

# wide -> long
long = sub.melt(id_vars=["item_id", "store_id"], value_vars=day_cols,
                var_name="d", value_name="units")
long = long.merge(calendar[["d", "date"]], on="d")
print("sales rows:", len(long))   # expect ~580k

# write the three tables
con = sqlite3.connect("forecast.db")
long[["item_id", "store_id", "date", "units"]].to_sql(
    "sales", con, if_exists="replace", index=False)
calendar[["d", "date", "wm_yr_wk", "weekday", "event_name_1", "snap_CA"]].to_sql(
    "calendar", con, if_exists="replace", index=False)
prices[prices["store_id"].isin(STORES) & prices["item_id"].isin(top_items)].to_sql(
    "prices", con, if_exists="replace", index=False)

# indexes: what makes per-product queries fast
con.execute("CREATE INDEX IF NOT EXISTS idx_sales ON sales(item_id, store_id, date)")
con.execute("CREATE INDEX IF NOT EXISTS idx_prices ON prices(item_id, store_id, wm_yr_wk)")
con.commit()
print("db built")

sales rows: 582300
db built


In [3]:
FEATURE_SQL = """
WITH daily AS (
    SELECT s.item_id, s.store_id, s.date, s.units,
           c.event_name_1, c.snap_CA, c.wm_yr_wk
    FROM sales s
    JOIN calendar c ON s.date = c.date
)
SELECT d.item_id, d.store_id, d.date, d.units,
       CAST(strftime('%w', d.date) AS INT)            AS dayofweek,
       CAST(strftime('%m', d.date) AS INT)            AS month,
       CASE WHEN d.event_name_1 IS NOT NULL THEN 1 ELSE 0 END AS is_event,
       d.snap_CA,
       LAG(d.units, 7)  OVER w                        AS lag_7,
       LAG(d.units, 14) OVER w                        AS lag_14,
       LAG(d.units, 28) OVER w                        AS lag_28,
       AVG(d.units) OVER (w ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING)  AS roll_mean_7,
       AVG(d.units) OVER (w ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS roll_mean_28,
       p.sell_price
FROM daily d
LEFT JOIN prices p
  ON p.item_id = d.item_id AND p.store_id = d.store_id AND p.wm_yr_wk = d.wm_yr_wk
WINDOW w AS (PARTITION BY d.item_id, d.store_id ORDER BY d.date)
ORDER BY d.item_id, d.store_id, d.date
"""

feat_sql = pd.read_sql(FEATURE_SQL, con, parse_dates=["date"])
print(feat_sql.shape)
feat_sql.head()

(582300, 14)


,item_id,store_id,date,units,dayofweek,month,is_event,snap_CA,lag_7,lag_14,lag_28,roll_mean_7,roll_mean_28,sell_price
0,FOODS_1_004,CA_1,2011-01-29,0,6,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,FOODS_1_004,CA_1,2011-01-30,0,0,1,0,0,NaN,NaN,NaN,0.0,0.0,NaN
2,FOODS_1_004,CA_1,2011-01-31,0,1,1,0,0,NaN,NaN,NaN,0.0,0.0,NaN
3,FOODS_1_004,CA_1,2011-02-01,0,2,2,0,1,NaN,NaN,NaN,0.0,0.0,NaN
4,FOODS_1_004,CA_1,2011-02-02,0,3,2,0,1,NaN,NaN,NaN,0.0,0.0,NaN


In [4]:
feat2 = pd.read_csv("app_data.csv", parse_dates=["date"])

In [5]:
chk = feat_sql[(feat_sql["item_id"] == "FOODS_3_090") & (feat_sql["store_id"] == "CA_1")].tail(5)
old = feat2[feat2["item_id"] == "FOODS_3_090"].tail(5)
print(chk[["date", "units", "lag_7", "roll_mean_7"]])
print(old[["date", "units", "lag_7", "roll_mean_7"]])

            date  units  lag_7  roll_mean_7
95104 2016-05-18     39   30.0    49.000000
95105 2016-05-19     51   35.0    50.285714
95106 2016-05-20     69   77.0    52.571429
95107 2016-05-21     67   47.0    51.428571
95108 2016-05-22     64   74.0    54.285714
           date  units  lag_7  roll_mean_7
5734 2016-05-18     39   30.0    49.000000
5735 2016-05-19     51   35.0    50.285714
5736 2016-05-20     69   77.0    52.571429
5737 2016-05-21     67   47.0    51.428571
5738 2016-05-22     64   74.0    54.285714


In [6]:
import os
print(f"forecast.db: {os.path.getsize('forecast.db') / 1e6:.1f} MB")

forecast.db: 48.7 MB


In [7]:
df = feat_sql.dropna(subset=["lag_7", "lag_14", "lag_28",
                             "roll_mean_7", "roll_mean_28", "sell_price"]).copy()

# price-derived features (same as before, now per store-item)
g = df.groupby(["item_id", "store_id"])["sell_price"]
df["price_lag_7"] = g.shift(7)
df["price_change"] = (df["sell_price"] - df["price_lag_7"]) / df["price_lag_7"]
df["price_rel"] = df["sell_price"] / g.transform("mean")
df = df.dropna(subset=["price_change"])

# categoricals
df["category"] = df["item_id"].str.split("_").str[0]
for c in ["item_id", "store_id", "category"]:
    df[c] = df[c].astype("category")
df["is_weekend"] = (df["dayofweek"].isin([0, 6])).astype(int)   # sqlite: 0=Sun, 6=Sat

FEATURES3 = ["dayofweek", "is_weekend", "month", "is_event", "snap_CA",
             "lag_7", "lag_14", "lag_28", "roll_mean_7", "roll_mean_28",
             "sell_price", "price_change", "price_rel",
             "store_id", "category", "item_id"]

print(df.shape)
df[FEATURES3].dtypes

(534924, 19)


C:\Users\kianz\AppData\Local\Temp\ipykernel_18252\88585649.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["category"] = df["item_id"].str.split("_").str[0]
C:\Users\kianz\AppData\Local\Temp\ipykernel_18252\88585649.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[c] = df[c].astype("category")
C:\Users\kianz\AppData\Local\Temp\ipykernel_18252\88585649.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = va

dayofweek          int64
is_weekend         int64
month              int64
is_event           int64
snap_CA            int64
lag_7            float64
lag_14           float64
lag_28           float64
roll_mean_7      float64
roll_mean_28     float64
sell_price       float64
price_change     float64
price_rel        float64
store_id        category
category        category
item_id         category
dtype: object

In [8]:
import lightgbm as lgb

cutoff3 = df["date"].max() - pd.Timedelta(28, unit="D")
train3 = df[df["date"] <= cutoff3]
test3  = df[df["date"] >  cutoff3]
print(f"train rows: {len(train3):,}  test rows: {len(test3):,}")

qmodels3 = {}
for q in (0.1, 0.5, 0.9):
    m = lgb.LGBMRegressor(objective="quantile", alpha=q,
                          n_estimators=800, learning_rate=0.05,
                          num_leaves=63, random_state=42)
    m.fit(train3[FEATURES3], train3["units"],
          categorical_feature=["store_id", "category", "item_id"])
    qmodels3[q] = m
    print(f"q{int(q*100)} trained")

train rows: 526,524  test rows: 8,400
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010432 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1924
[LightGBM] [Info] Number of data points in the train set: 526524, number of used features: 16
q10 trained
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018734 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1924
[LightGBM] [Info] Number of data points in the train set: 526524, number of used features: 16
[LightGBM] [Info] Start training from score 9.000000


KeyboardInterrupt: 

In [ ]:
test3 = test3.copy()
test3["q50"] = qmodels3[0.5].predict(test3[FEATURES3]).clip(min=0)
test3["q10"] = qmodels3[0.1].predict(test3[FEATURES3]).clip(min=0)
test3["q90"] = qmodels3[0.9].predict(test3[FEATURES3]).clip(min=0)

mae3 = test3.groupby(["item_id", "store_id"], observed=True).apply(
    lambda x: np.mean(np.abs(x["units"] - x["q50"]))).mean()

# seasonal naive across all 300 series
naive_maes = []
for (it, stq), gdf in test3.groupby(["item_id", "store_id"], observed=True):
    tr_units = train3[(train3["item_id"] == it) & (train3["store_id"] == stq)] \
                   .sort_values("date")["units"].values
    gdf = gdf.sort_values("date")
    reps = int(np.ceil(len(gdf) / 7))
    nf = np.tile(tr_units[-7:], reps)[:len(gdf)]
    naive_maes.append(np.mean(np.abs(gdf["units"].values - nf)))
naive3 = np.mean(naive_maes)

cov3 = ((test3["units"] >= test3["q10"]) & (test3["units"] <= test3["q90"])).mean()

print(f"Series: {test3.groupby(['item_id','store_id'], observed=True).ngroups}")
print(f"Naive MAE:  {naive3:.2f}")
print(f"Model MAE:  {mae3:.2f}   improvement: {(naive3-mae3)/naive3:.1%}")
print(f"Coverage (nominal 80%): {cov3:.1%}")

In [ ]:
for q, m in qmodels3.items():
    m.booster_.save_model(f"model_q{int(q*100)}_v3.txt")
print("saved v3 models")